# 🌳 Capítulo 8 — Árboles (Trees)
## Notebook 3: Árboles Binarios — ADT, Implementación y Propiedades

**Libro:** Goodrich, Tamassia & Goldwasser — *Data Structures and Algorithms in Python*, §8.3  
**Materia:** Estructuras de Datos y Algoritmos — Lic. en Sistemas | **Año:** 2026

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap08/03_Arboles_Binarios_Teoria.ipynb)

## 🎯 Objetivos de aprendizaje

1. Definir el **ADT Árbol Binario** y sus operaciones adicionales respecto al árbol general.
2. Comprender e implementar **LinkedBinaryTree** con representación enlazada.
3. Comprendre la representación **basada en arreglos** (nivel-numeración) y sus ventajas.
4. Verificar numéricamente las **propiedades fundamentales** de los árboles binarios.
5. Comparar ambas implementaciones por espacio, tiempo y flexibilidad.

## 📖 Contenidos

| Sección | Tema |
|---------|------|
| §8.3.1 | ADT de árbol binario |
| §8.3.2 | LinkedBinaryTree — implementación enlazada |
| §8.3.3 | Representación basada en arreglos |
| §8.3.4 | Propiedades de los árboles binarios |
| §8.3.5 | Comparación y análisis de desempeño |

## 🔧 Configuración del entorno

In [ ]:
import sys
from abc import ABCMeta, abstractmethod

print(f"Python {sys.version}")

# ── ADT Tree (base) ───────────────────────────────────────────────────────────
class Tree(metaclass=ABCMeta):
    class Position(metaclass=ABCMeta):
        @abstractmethod
        def element(self): raise NotImplementedError
        @abstractmethod
        def __eq__(self, other): raise NotImplementedError
        def __ne__(self, other): return not (self == other)

    @abstractmethod
    def root(self): raise NotImplementedError
    @abstractmethod
    def parent(self, p): raise NotImplementedError
    @abstractmethod
    def num_children(self, p): raise NotImplementedError
    @abstractmethod
    def children(self, p): raise NotImplementedError
    @abstractmethod
    def __len__(self): raise NotImplementedError

    def is_root(self, p): return self.root() == p
    def is_leaf(self, p): return self.num_children(p) == 0
    def is_empty(self): return len(self) == 0

    def depth(self, p):
        return 0 if self.is_root(p) else 1 + self.depth(self.parent(p))

    def height(self, p=None):
        if p is None: p = self.root()
        return 0 if self.is_leaf(p) else 1 + max(self.height(c) for c in self.children(p))

print("ADT Tree cargado ✓")

---

## §8.3.1 ADT de Árbol Binario

Un **árbol binario** es un árbol en el que cada nodo tiene **como máximo 2 hijos**, llamados hijo **izquierdo** y **derecho**.

Un árbol binario es **propio** (proper / full) si cada nodo es hoja o tiene exactamente 2 hijos.  
Un árbol binario **no propio** (improper) puede tener un nodo con sólo un hijo.

### Operaciones adicionales al ADT Tree

| Método | Descripción |
|--------|-------------|
| `T.left(p)` | Retorna el hijo izquierdo de p (o None) |
| `T.right(p)` | Retorna el hijo derecho de p (o None) |
| `T.sibling(p)` | Retorna el hermano de p (o None) |

Estos métodos se **añaden** a los heredados del ADT Tree general.

In [ ]:
# ── ADT BinaryTree ────────────────────────────────────────────────────────────
class BinaryTree(Tree):
    """Abstract base class for a binary tree structure (§8.3.1)."""

    # -- additional abstract methods --
    @abstractmethod
    def left(self, p):
        """Return a Position representing p's left child (or None)."""
        raise NotImplementedError('must be implemented by subclass')

    @abstractmethod
    def right(self, p):
        """Return a Position representing p's right child (or None)."""
        raise NotImplementedError('must be implemented by subclass')

    # -- concrete method defined by convention here --
    def sibling(self, p):
        """Return a Position representing p's sibling (or None if no sibling)."""
        parent = self.parent(p)
        if parent is None:          # p must be the root
            return None
        if p == self.left(parent):
            return self.right(parent)  # possibly None
        else:
            return self.left(parent)   # possibly None

    def children(self, p):
        """Generate an iteration of Positions representing p's children."""
        if self.left(p) is not None:
            yield self.left(p)
        if self.right(p) is not None:
            yield self.right(p)

    def num_children(self, p):
        """Return the number of children that Position p has."""
        count = 0
        if self.left(p) is not None:
            count += 1
        if self.right(p) is not None:
            count += 1
        return count

    # -- inorder traversal --
    def inorder(self):
        if not self.is_empty():
            for p in self._subtree_inorder(self.root()):
                yield p

    def _subtree_inorder(self, p):
        if self.left(p) is not None:
            for other in self._subtree_inorder(self.left(p)):
                yield other
        yield p
        if self.right(p) is not None:
            for other in self._subtree_inorder(self.right(p)):
                yield other

    def positions(self):
        return self.inorder()

print("ADT BinaryTree definido ✓")
print("Métodos concretos: sibling, children, num_children, inorder")

---

## §8.3.2 LinkedBinaryTree — Implementación Enlazada

La implementación **enlazada** usa nodos con tres punteros: `parent`, `left`, `right`.

```
   ┌───────────┐
   │  _parent  │──► nodo padre
   │─────────  │
   │  _element │     el dato
   │─────────  │
   │  _left    │──► hijo izquierdo
   │  _right   │──► hijo derecho
   └───────────┘
```

**Clase interna `_Node`** — `__slots__` para eficiencia de memoria.  
**Clase interna `_Position`** — envuelve un nodo para validar pertenencia al árbol correcto.

In [ ]:
class LinkedBinaryTree(BinaryTree):
    """Linked representation of a binary tree structure (§8.3.2)."""

    class _Node:
        __slots__ = '_element', '_parent', '_left', '_right'
        def __init__(self, element, parent=None, left=None, right=None):
            self._element = element
            self._parent  = parent
            self._left    = left
            self._right   = right

    class _Position(BinaryTree.Position):
        """An abstraction representing the location of a single element."""
        def __init__(self, container, node):
            self._container = container
            self._node      = node
        def element(self):
            return self._node._element
        def __eq__(self, other):
            return type(other) is type(self) and other._node is self._node

    def _validate(self, p):
        """Return associated node, if position is valid."""
        if not isinstance(p, self._Position):
            raise TypeError('p must be proper Position type')
        if p._container is not self:
            raise ValueError('p does not belong to this container')
        if p._node._parent is p._node:   # convention for deprecated nodes
            raise ValueError('p is no longer valid')
        return p._node

    def _make_position(self, node):
        """Return Position instance for given node (or None if no node)."""
        return self._Position(self, node) if node is not None else None

    # ── Constructor ───────────────────────────────────────────────────────────
    def __init__(self):
        """Create an initially empty binary tree."""
        self._root = None
        self._size = 0

    # ── Public accessors ──────────────────────────────────────────────────────
    def __len__(self):
        return self._size

    def root(self):
        return self._make_position(self._root)

    def parent(self, p):
        node = self._validate(p)
        return self._make_position(node._parent)

    def left(self, p):
        node = self._validate(p)
        return self._make_position(node._left)

    def right(self, p):
        node = self._validate(p)
        return self._make_position(node._right)

    # ── Nonpublic update methods ──────────────────────────────────────────────
    def add_root(self, e):
        """Place element e at the root of an empty tree and return new Position.
        Raise ValueError if tree nonempty.
        """
        if self._root is not None:
            raise ValueError('Root exists')
        self._size = 1
        self._root = self._Node(e)
        return self._make_position(self._root)

    def add_left(self, p, e):
        """Create a new left child for Position p, storing element e.
        Return the Position of new node.
        Raise ValueError if Position p is invalid or p already has a left child.
        """
        node = self._validate(p)
        if node._left is not None:
            raise ValueError('Left child exists')
        self._size += 1
        node._left  = self._Node(e, parent=node)
        return self._make_position(node._left)

    def add_right(self, p, e):
        """Create a new right child for Position p, storing element e.
        Return the Position of new node.
        Raise ValueError if Position p is invalid or p already has a right child.
        """
        node = self._validate(p)
        if node._right is not None:
            raise ValueError('Right child exists')
        self._size += 1
        node._right = self._Node(e, parent=node)
        return self._make_position(node._right)

    def replace(self, p, e):
        """Replace the element at position p with e, and return old element."""
        node = self._validate(p)
        oldvalue = node._element
        node._element = e
        return oldvalue

    def delete(self, p):
        """Delete the node at Position p, and replace it with its child, if any.
        Return the element formerly stored at Position p.
        Raise ValueError if Position p is invalid or p has two children.
        """
        node = self._validate(p)
        if node._left is not None and node._right is not None:
            raise ValueError('Position has two children')
        child = node._left if node._left else node._right   # might be None
        if child is not None:
            child._parent = node._parent   # child's grandparent becomes parent
        if node is self._root:
            self._root = child             # child becomes root
        else:
            parent = node._parent
            if node is parent._left:
                parent._left = child
            else:
                parent._right = child
        self._size -= 1
        node._parent = node               # convention for deprecated node
        return node._element

    def attach(self, p, t1, t2):
        """Attach trees t1 and t2, respectively, as the left and right subtrees
        of the external Position p. As a side effect, set t1 and t2 to empty.
        Raise TypeError if trees t1 and t2 do not match type of this tree.
        Raise ValueError if Position p is invalid or not external.
        """
        node = self._validate(p)
        if not self.is_leaf(p):
            raise ValueError('position must be leaf')
        if not type(self) is type(t1) is type(t2):
            raise TypeError('Tree types must match')
        self._size += len(t1) + len(t2)
        if not t1.is_empty():
            t1._root._parent = node
            node._left = t1._root
            t1._root = None
            t1._size = 0
        if not t2.is_empty():
            t2._root._parent = node
            node._right = t2._root
            t2._root = None
            t2._size = 0

print("LinkedBinaryTree implementado ✓")
print("Métodos: add_root, add_left, add_right, replace, delete, attach")

---

### Demostración de uso de LinkedBinaryTree

Construimos el árbol de expresión aritmética: `((3+1) * (9-5))`

```
         *
        / \
       +   -
      / \ / \
     3  1 9  5
```

In [ ]:
# Construir árbol de expresión ((3+1) * (9-5))
T = LinkedBinaryTree()
r = T.add_root('*')
left  = T.add_left(r,    '+')
right = T.add_right(r,   '-')
T.add_left(left,   '3')
T.add_right(left,  '1')
T.add_left(right,  '9')
T.add_right(right, '5')

print(f"Tamaño del árbol: {len(T)} nodos")
print(f"Altura:           {T.height()}")
print(f"Profundidad de '3': {T.depth(T.left(T.left(T.root())))}")
print()

# Recorrido inorden (expresa la expresión en notación infija)
inorden = [p.element() for p in T.inorder()]
print("Inorden (notación infija):", " ".join(inorden))

# Recorrido preorden (notación prefija)
def preorder(T, p=None):
    if p is None: p = T.root()
    yield p.element()
    for c in T.children(p): yield from preorder(T, c)

preorden = list(preorder(T))
print("Preorden (notación prefija):", " ".join(preorden))

# Verificar sibling
plus_node = T.left(T.root())
minus_node = T.right(T.root())
print()
print(f"siblings: sibling('+') = '{T.sibling(plus_node).element()}' ✓")
print(f"siblings: sibling('-') = '{T.sibling(minus_node).element()}' ✓")

---

## §8.3.3 Representación Basada en Arreglos

**Nivel-numeración (level numbering):** función `f(p)` que asigna un índice entero a cada posición:

- Si p es raíz: `f(p) = 0`
- Si p es hijo izquierdo de q: `f(p) = 2·f(q) + 1`
- Si p es hijo derecho de q: `f(p) = 2·f(q) + 2`

```
         A (f=0)
        /       \
     B (f=1)   C (f=2)
    /    \
 D (f=3) E (f=4)
```

**Ventajas:** acceso O(1) a padre e hijos (`(f-1)//2`, `2f+1`, `2f+2`)  
**Desventaja:** árbol desbalanceado puede desperdiciar O(2ⁿ) espacio

In [ ]:
class ArrayBinaryTree:
    """Array-based (level-numbering) binary tree — read-only demo."""

    def __init__(self, array):
        """array[i] = element at f(p) = i; None means that slot is empty."""
        self._data = array[:]

    def __len__(self):
        return sum(1 for x in self._data if x is not None)

    @staticmethod
    def parent_index(i): return (i - 1) // 2 if i > 0 else None
    @staticmethod
    def left_index(i):   return 2 * i + 1
    @staticmethod
    def right_index(i):  return 2 * i + 2

    def element(self, i):
        return self._data[i] if i < len(self._data) else None

    def depth(self, i):
        count = 0
        while i > 0:
            i = self.parent_index(i)
            count += 1
        return count

    def height(self):
        max_d = 0
        for i in range(len(self._data)):
            if self._data[i] is not None:
                if self.left_index(i) >= len(self._data) or self._data[self.left_index(i)] is None:
                    if self.right_index(i) >= len(self._data) or self._data[self.right_index(i)] is None:
                        max_d = max(max_d, self.depth(i))
        return max_d

    def visualize(self):
        print("Índice  | Elemento | Padre | Izq | Der")
        print("--------|----------|-------|-----|----")
        for i, val in enumerate(self._data):
            if val is not None:
                pi = self.parent_index(i)
                li = self.left_index(i)
                ri = self.right_index(i)
                p_val = self._data[pi] if pi is not None else '-'
                l_val = self._data[li] if li < len(self._data) and self._data[li] else '-'
                r_val = self._data[ri] if ri < len(self._data) and self._data[ri] else '-'
                print(f"  f={i:<3}  |  {str(val):<7} | {str(p_val):<5} | {str(l_val):<3} | {str(r_val)}")

# Árbol de ejemplo:
#        A(0)
#       /    \
#     B(1)   C(2)
#    /   \
#  D(3)  E(4)
arr_tree = ArrayBinaryTree(['A','B','C','D','E',None,None])
arr_tree.visualize()
print()
print(f"Altura del árbol: {arr_tree.height()}")
print(f"Profundidad de D (i=3): {arr_tree.depth(3)}")

---

## §8.3.4 Propiedades Fundamentales de los Árboles Binarios

Sea n = número total de nodos, nₑ = nodos externos (hojas), nᵢ = nodos internos, h = altura.

**Proposición 8.7** (cualquier árbol binario con n ≥ 1 nodos):
$$h + 1 \leq n \leq 2^{h+1} - 1$$

**Proposición 8.8** (cualquier árbol binario):
$$1 \leq n_E \leq 2^h$$

**Proposición 8.9** (cualquier árbol binario):
$$h \leq n_I \leq 2^h - 1$$

**Proposición 8.10** (árbol binario **propio**):
- $n_E = n_I + 1$
- $2h + 1 \leq n \leq 2^{h+1} - 1$
- $\frac{n+1}{2} \leq n_E \leq 2^h$
- $\frac{n-1}{2} \leq n_I \leq 2^h - 1$
- $\log_2(n+1) - 1 \leq h \leq \frac{n-1}{2}$

In [ ]:
import math

def count_nodes(T, p=None):
    """Devuelve (n_externo, n_interno) sobre el subárbol de p."""
    if p is None: p = T.root()
    if T.is_leaf(p):
        return 1, 0
    ext, int_ = 0, 1
    for c in T.children(p):
        e, i = count_nodes(T, c)
        ext += e; int_ += i
    return ext, int_

def verificar_propiedades(T, nombre="T"):
    n   = len(T)
    nE, nI = count_nodes(T)
    h   = T.height()
    n_total = nE + nI

    print(f"=== {nombre} ===")
    print(f"  n={n_total}, nE={nE}, nI={nI}, h={h}")
    print()

    # Prop 8.7
    p87_ok = (h + 1 <= n_total <= 2**(h+1) - 1)
    print(f"  Prop 8.7: h+1 <= n <= 2^(h+1)-1")
    print(f"    {h+1} <= {n_total} <= {2**(h+1)-1}  →  {'✅' if p87_ok else '❌'}")

    # Prop 8.10 (solo si propio)
    proper = all(T.num_children(p) != 1 for p in (T.root() if hasattr(T,'root') else []))
    leaves_only = all(T.num_children(p) in (0,2) for p in T.positions())
    if leaves_only:
        p810_ok = (nE == nI + 1)
        print(f"  Prop 8.10 (árbol propio): nE = nI + 1")
        print(f"    {nE} = {nI} + 1  →  {'✅' if p810_ok else '❌'}")
    else:
        print("  Árbol no es propio (tiene nodos con 1 hijo)")
    print()

# Árbol propio: expresión ((3+1) * (9-5))
verificar_propiedades(T, "Árbol expresión ((3+1)*(9-5))")

# Árbol completo de altura 3
T2 = LinkedBinaryTree()
r2 = T2.add_root(1)
a = T2.add_left(r2, 2); b = T2.add_right(r2, 3)
c = T2.add_left(a, 4); d = T2.add_right(a, 5)
e = T2.add_left(b, 6); f = T2.add_right(b, 7)
verificar_propiedades(T2, "Árbol completo altura 2")

---

## 📊 Comparación: Implementación Enlazada vs. Basada en Arreglos

| Criterio | LinkedBinaryTree | ArrayBinaryTree |
|----------|-----------------|-----------------|
| Espacio (árbol balanceado) | O(n) | O(n) |
| Espacio (árbol degenerado) | O(n) | O(2ⁿ) — desperdicioso |
| Acceso a hijo | O(1) por puntero | O(1) por cálculo de índice |
| Inserción / eliminación | O(1) amortizado | Complejo (puede requerir desplazamientos) |
| Uso típico | Árboles generales, BST, expresiones | Heaps, torneos, árboles completos |
| Soporte para `attach` | ✅ O(1) | ❌ No directo |

**Conclusión:** Usar arreglos cuando el árbol es **completo o casi completo** (p.ej. Heaps del cap. 9).  
Para árboles dinámicos o no balanceados, preferir **representación enlazada**.

## 🧪 Tests

In [ ]:
import unittest

class TestLinkedBinaryTree(unittest.TestCase):

    def setUp(self):
        """
              1
             / \\
            2   3
           / \\
          4   5
        """
        self.T = LinkedBinaryTree()
        self.r  = self.T.add_root(1)
        self.L  = self.T.add_left(self.r, 2)
        self.R  = self.T.add_right(self.r, 3)
        self.LL = self.T.add_left(self.L, 4)
        self.LR = self.T.add_right(self.L, 5)

    def test_size(self):
        self.assertEqual(len(self.T), 5)

    def test_height(self):
        self.assertEqual(self.T.height(), 2)

    def test_depth(self):
        self.assertEqual(self.T.depth(self.r), 0)
        self.assertEqual(self.T.depth(self.L), 1)
        self.assertEqual(self.T.depth(self.LL), 2)

    def test_leaves(self):
        leaves = [p.element() for p in self.T.positions() if self.T.is_leaf(p)]
        self.assertIn(4, leaves)
        self.assertIn(5, leaves)
        self.assertIn(3, leaves)
        self.assertEqual(len(leaves), 3)

    def test_inorder(self):
        result = [p.element() for p in self.T.inorder()]
        self.assertEqual(result, [4, 2, 5, 1, 3])

    def test_sibling(self):
        self.assertEqual(self.T.sibling(self.L).element(), 3)
        self.assertEqual(self.T.sibling(self.LL).element(), 5)
        self.assertIsNone(self.T.sibling(self.r))   # root has no sibling

    def test_property_nE_eq_nI_plus_1(self):
        """Árbol propio: nE = nI + 1"""
        nE, nI = count_nodes(self.T)
        self.assertEqual(nE, nI + 1)

    def test_replace(self):
        old = self.T.replace(self.L, 99)
        self.assertEqual(old, 2)
        self.assertEqual(self.L.element(), 99)

    def test_delete(self):
        """Borrar hoja LL (4) debe reducir tamaño a 4."""
        self.T.delete(self.LL)
        self.assertEqual(len(self.T), 4)

    def test_add_root_twice_raises(self):
        with self.assertRaises(ValueError):
            self.T.add_root(99)

    def test_add_left_twice_raises(self):
        with self.assertRaises(ValueError):
            self.T.add_left(self.L, 99)

suite = unittest.TestLoader().loadTestsFromTestCase(TestLinkedBinaryTree)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\n{'✅ Todos los tests pasaron' if result.wasSuccessful() else '❌ Hubo fallos'}")

---

## ✅ Autoevaluación

- [ ] Dibujar un árbol binario propio de altura 3 con el mínimo número de nodos
- [ ] Verificar que nE = nI + 1 para ese árbol
- [ ] Implementar, sin ver el código, el método `sibling(p)` de `BinaryTree`
- [ ] Calcular los índices de arreglo para los nodos del árbol anterior
- [ ] Explicar por qué un árbol degenerado con representación de arreglo usa O(2ⁿ) espacio

## 📚 Referencias

- Goodrich et al., §8.3 — Binary Trees
- §8.3.2 — LinkedBinaryTree
- §8.3.3 — Array-Based Representation of a Binary Tree

---
*Capítulo 8 — Árboles | Estructuras de Datos y Algoritmos — Lic. en Sistemas*  
*Material bajo licencia [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)*